# molprop-ensemble — Bootstrap Kaggle

Notebook ini: clone repo dari GitHub, install dependency, cek GPU, lalu jalankan **percobaan kecil**
(1 dataset x 1 seed) untuk memastikan pipeline jalan sebelum full run 3 dataset x 5 seed.

**Sebelum Run All:**
- Accelerator (panel kanan) -> `GPU T4 x2`
- Internet (panel kanan) -> `On`
- **Repo ini PRIVATE** -> wajib siapkan token dulu, kalau tidak sel clone akan gagal/macet:
  1. Buat token: https://github.com/settings/tokens -> *Generate new token (classic)* -> centang scope `repo` -> Generate -> copy.
  2. Di notebook Kaggle: **Add-ons -> Secrets** -> tambah secret baru, Label = `GH_TOKEN`, Value = token tsb -> pastikan **Attached** ke notebook ini.
  3. Baru jalankan sel 1.

Urutan sel: 1) clone -> 2) install -> 3) cek GPU -> 4) Fase 1 (data) ->
5) Fase 4 percobaan kecil (RF + ChemBERTa + D-MPNN, 1 dataset x 1 seed) ->
6) Fase 5-7 percobaan kecil -> 7) (opsional) full run semua dataset x seed.

Lihat `KAGGLE.md` di repo untuk detail & troubleshooting.

## 1. Clone repo dari GitHub

In [ ]:
REPO_OWNER = "belvahector-ship-it"
REPO_NAME = "pharm_"
REPO_DIR = "pharm_"

import os
import subprocess

if not os.path.exists(REPO_DIR):
    # Repo PRIVATE -> WAJIB token, kalau tidak `git clone` menggantung menunggu
    # username/password yang tidak bisa dijawab dari sel notebook.
    # Simpan token di Add-ons -> Secrets dgn label GH_TOKEN (lihat KAGGLE.md).
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass

    if token:
        url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
        print("clone via GH_TOKEN (repo private)...")
    else:
        url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
        print("GH_TOKEN tidak ditemukan -> mencoba clone publik biasa.")
        print("Kalau repo masih PRIVATE, sel ini akan gagal (bukan menggantung).")
        print("Perbaikan: Add-ons > Secrets > tambah GH_TOKEN, lalu Run sel ini lagi.")

    # timeout + stdin=DEVNULL supaya kalau tetap butuh auth interaktif, sel berhenti
    # dgn error jelas alih-alih 'running' selamanya.
    result = subprocess.run(
        ["git", "clone", url, REPO_DIR],
        capture_output=True, text=True, timeout=120,
        stdin=subprocess.DEVNULL,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "Clone gagal. Kemungkinan besar repo PRIVATE tanpa GH_TOKEN yang valid. "
            "Lihat instruksi di atas / KAGGLE.md bagian Secrets."
        )
else:
    print(f"{REPO_DIR} sudah ada, skip clone. (Untuk update: cd {REPO_DIR} && git pull)")

%cd {REPO_DIR}
!git log --oneline -3

## 2. Install dependency

In [ ]:
# Kaggle sudah menyediakan torch (jangan dipaksa versi lain, biar cocok dgn CUDA image-nya).
# chemprop 1.x TIDAK ada untuk Python modern (rilis 1.x terakhir mensyaratkan Python <3.9),
# jadi kita pakai chemprop 2.x -- dmpnn_model.py sudah disesuaikan ke CLI `chemprop train/predict`.
# deepchem TIDAK diperlukan -- data_loader.py sudah download CSV mentah MoleculeNet langsung.
!pip install -q rdkit "chemprop>=2.1,<3.0" transformers
print("install selesai")

## 3. Cek environment (GPU x2, library)

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
print("jumlah GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

assert torch.cuda.device_count() >= 2, (
    "Hanya terdeteksi <2 GPU. Set Accelerator ke 'GPU T4 x2' di panel kanan notebook, lalu restart."
)

import rdkit, sklearn, transformers
print("rdkit:", rdkit.__version__)
print("sklearn:", sklearn.__version__)
print("transformers:", transformers.__version__)
import chemprop
print("chemprop:", chemprop.__version__)

In [ ]:
# (Perbaikan audit) Algoritma scaffold split BERUBAH menjadi deterministik. Artefak dari run
# LAMA (split cache, prediksi, checkpoint) jadi tidak konsisten dgn split baru. Set True SEKALI
# setelah update ini untuk membersihkan, lalu boleh kembalikan ke False.
RESET_ARTIFACTS = False

import os, shutil
if RESET_ARTIFACTS:
    for d in ["data/splits", "outputs/predictions", "outputs/checkpoints", "outputs/results"]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
    print("Artefak lama dibersihkan. Kembalikan RESET_ARTIFACTS=False untuk run berikutnya.")
else:
    print("RESET_ARTIFACTS=False (dilewati). Jika baru update kode & split berubah, set True sekali.")

## 4. Fase 1 — Prepare data & scaffold split
Split dibuat sekali (fixed), dipakai ulang persis oleh RF/ChemBERTa/D-MPNN (Bagian 3 blueprint).

In [ ]:
# (K1/K2) Diagnostik mutu scaffold split: WAJIB "overlap SCAFFOLD = 0" & "bocor->train = 0.0%".
# Lihat juga rasio singleton (wajar tinggi utk MoleculeNet). Ini menjawab kecurigaan audit soal
# AUC yang terlalu tinggi (split mendekati random). Kalau STATUS OK -> split sah.
!python scripts/00_diagnose_split.py

In [ ]:
!python scripts/01_prepare_data.py

## 5. Percobaan kecil — 1 dataset x 1 seed
Verifikasi pipeline utuh berjalan (train -> TTA -> fusion -> evaluasi) sebelum full run yang mahal.
Dataset `bbbp`, seed `0` saja. ChemBERTa & D-MPNN dijalankan **paralel di GPU 0 & GPU 1**, RF di CPU.

In [ ]:
import subprocess

TRIAL_DATASETS = ["bbbp"]
TRIAL_SEEDS = ["0"]

cmds = {
    "chemberta": ["python", "scripts/02_train_baselines.py", "--model", "chemberta",
                  "--datasets", *TRIAL_DATASETS, "--seeds", *TRIAL_SEEDS],
    "dmpnn": ["python", "scripts/02_train_baselines.py", "--model", "dmpnn",
              "--datasets", *TRIAL_DATASETS, "--seeds", *TRIAL_SEEDS],
    "rf": ["python", "scripts/02_train_baselines.py", "--model", "rf",
           "--datasets", *TRIAL_DATASETS, "--seeds", *TRIAL_SEEDS],
}

procs = {name: subprocess.Popen(cmd) for name, cmd in cmds.items()}
for name, p in procs.items():
    rc = p.wait()
    print(f"[{name}] selesai, return code = {rc}")
    assert rc == 0, f"{name} gagal (return code {rc}) — cek log di atas / outputs/logs/"

In [ ]:
# Fase 5 — TTA ChemBERTa (val & test), khusus dataset/seed percobaan
!python scripts/04_run_tta.py --datasets {" ".join(TRIAL_DATASETS)} --seeds {" ".join(TRIAL_SEEDS)}

In [ ]:
# Fase 6 — fusion (avg / weighted / weighted_tta / stacking)
!python scripts/03_run_fusion.py --datasets {" ".join(TRIAL_DATASETS)} --seeds {" ".join(TRIAL_SEEDS)}

In [ ]:
# Fase 7 — evaluasi (tabel hasil percobaan kecil, n_seed=1 -> std=0, p-value tidak bermakna,
# ini HANYA untuk memastikan pipeline jalan end-to-end, bukan hasil final)
!python scripts/05_evaluate.py --datasets {" ".join(TRIAL_DATASETS)} --seeds {" ".join(TRIAL_SEEDS)}

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/results/final_table.csv")
df

**Kalau semua sel di atas selesai tanpa error dan tabel muncul** -> pipeline utuh terbukti jalan.
Lanjut ke Full Run di bawah (mahal, ~9-12 jam tergantung sesi Kaggle; checkpoint & resume otomatis aktif).

## 6. (Opsional) Full run — semua dataset x semua seed
Jangan jalankan sel ini sebelum percobaan kecil di atas sukses.
Sesi bisa terputus; checkpoint per epoch + resume otomatis sudah aktif (`config.CHECKPOINT`)
-> tinggal jalankan ulang sel ini, training lanjut dari epoch terakhir.

In [ ]:
RUN_FULL = False  # ubah ke True untuk menjalankan full run

if RUN_FULL:
    import subprocess
    procs = {
        "chemberta": subprocess.Popen(["python", "scripts/02_train_baselines.py", "--model", "chemberta"]),
        "dmpnn": subprocess.Popen(["python", "scripts/02_train_baselines.py", "--model", "dmpnn"]),
        "rf": subprocess.Popen(["python", "scripts/02_train_baselines.py", "--model", "rf"]),
    }
    for name, p in procs.items():
        rc = p.wait()
        print(f"[{name}] selesai, return code = {rc}")
else:
    print("RUN_FULL=False, dilewati. Set RUN_FULL=True untuk full run 3 dataset x 5 seed.")

In [ ]:
# S2: rekam versi environment AKTUAL utk lampiran/reproducibility paper.
!pip freeze > outputs/results/pip_freeze.txt
print("pip freeze -> outputs/results/pip_freeze.txt")
# (05_evaluate juga menulis outputs/results/environment.txt berisi versi paket kunci)

!cd outputs && zip -rq /kaggle/working/hasil_outputs.zip .
print("tersimpan: /kaggle/working/hasil_outputs.zip")

## 7. Simpan hasil
`outputs/` di-gitignore (tidak ikut ke GitHub). Zip di sini lalu ambil dari panel Output Kaggle,
atau klik **Save Version** supaya seluruh `outputs/` tersimpan sebagai Output notebook.

In [ ]:
!cd outputs && zip -rq /kaggle/working/hasil_outputs.zip .
print("tersimpan: /kaggle/working/hasil_outputs.zip")